# OW-DETR Task 1 Training Notebook (IP102 - 25 Classes Split)
Notebook này thực thi **Task 1 (7 lớp)** cho bài toán Open World Object Detection (OW-DETR) trên bộ dữ liệu IP102 (25 lớp).
Sử dụng checkpoint Deformable DETR tại `/kaggle/input/models/chienkhu/deformable/pytorch/default/1/pytorch_model.bin`.

In [1]:
import os

# 1. Quay về thư mục làm việc Kaggle
%cd /kaggle/working

if not os.path.exists("/kaggle/working/OW-DETR"):
    !git clone https://github.com/nta2112/OW-DETR.git /kaggle/working/OW-DETR
else:
    print("Thư mục OW-DETR đã tồn tại. Đang đồng bộ bản mới nhất...")
    %cd /kaggle/working/OW-DETR
    !git pull origin main

%cd /kaggle/working/OW-DETR


/kaggle/working
Cloning into '/kaggle/working/OW-DETR'...
remote: Enumerating objects: 366, done.
remote: Counting objects: 100% (366/366), done.
remote: Compressing objects: 100% (222/222), done.
remote: Total 366 (delta 193), reused 300 (delta 127), pack-reused 0 (from 0)
Receiving objects: 100% (366/366), 2.09 MiB | 8.43 MiB/s, done.
Resolving deltas: 100% (193/193), done.
/kaggle/working/OW-DETR


In [2]:
%cd /kaggle/working/OW-DETR

# 2. Cài đặt các thư viện cần thiết
!pip install -r requirements.txt

# 3. Tải DINO ResNet-50 backbone pretrain
!mkdir -p models
!wget -N https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth -O models/dino_resnet50_pretrain.pth

# 4. Biên dịch các CUDA operators (MultiScaleDeformableAttention)
%cd models/ops
!sh make.sh
!python test.py

%cd /kaggle/working/OW-DETR


/kaggle/working/OW-DETR
for details.

--2026-07-21 08:42:59--  https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.180.94, 65.9.180.48, 65.9.180.116, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.180.94|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 94345885 (90M) [application/zip]
Saving to: ‘models/dino_resnet50_pretrain.pth’

models/dino_resnet5 100%[===================>]  89.97M   336MB/s    in 0.3s    

2026-07-21 08:42:59 (336 MB/s) - ‘models/dino_resnet50_pretrain.pth’ saved [94345885/94345885]

/kaggle/working/OW-DETR/models/ops
running build
running build_py
creating build/lib.linux-x86_64-cpython-312/modules
copying modules/__init__.py -> build/lib.linux-x86_64-cpython-312/modules
copying modules/ms_deform_attn.py -> build/lib.linux-x86_64-cpython-312/modules
creating build/lib.linux-x86_64-cpython-312/functions
copying functi

In [3]:
import torch
import os

# 5. Chuyển đổi trọng số từ checkpoint DETR (pytorch_model.bin) sang OW-DETR
input_path = "/kaggle/input/models/chienkhu/deformable/pytorch/default/1/pytorch_model.bin"
converted_path = "/kaggle/working/OW-DETR/models/deformable_detr_coco_converted.pth"

print("-> Đang ánh xạ trọng số DETR checkpoint tương thích với OW-DETR...")
os.makedirs("/kaggle/working/OW-DETR/models", exist_ok=True)

if not os.path.exists(converted_path):
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Không tìm thấy checkpoint tại {input_path}. Vui lòng kiểm tra Kaggle input.")
        
    state_dict = torch.load(input_path, map_location="cpu")
    new_state_dict = {}
    
    for k, v in state_dict.items():
        # Bỏ qua class_embed do số lượng lớp thay đổi
        if "class_embed." in k:
            continue
            
        new_k = k
        if k.startswith("model.backbone.conv_encoder.model."):
            new_k = k.replace("model.backbone.conv_encoder.model.", "backbone.0.body.")
        elif k.startswith("model.encoder."):
            new_k = k.replace("model.encoder.", "transformer.encoder.")
        elif k.startswith("model.decoder."):
            new_k = k.replace("model.decoder.", "transformer.decoder.")
        elif k == "model.level_embed":
            new_k = "transformer.level_embed"
        elif k.startswith("model.reference_points."):
            new_k = k.replace("model.reference_points.", "transformer.reference_points.")
        elif k == "model.query_position_embeddings.weight":
            new_k = "query_embed.weight"
            if v.shape[0] > 100:
                print(f"  [Đã cắt] query_embed.weight từ {list(v.shape)} xuống [100, 512]")
                v = v[:100, :]
        elif k.startswith("model."):
            new_k = k[6:]
            
        new_state_dict[new_k] = v

    torch.save({"model": new_state_dict, "epoch": -1}, converted_path)
    print(f"✓ Hoàn thành! Đã chuyển đổi checkpoint thành công: {converted_path}")
else:
    print("-> Checkpoint tương thích đã tồn tại, bỏ qua chuyển đổi.")


-> Đang ánh xạ trọng số DETR checkpoint tương thích với OW-DETR...
  [Đã cắt] query_embed.weight từ [300, 512] xuống [100, 512]
✓ Hoàn thành! Đã chuyển đổi checkpoint thành công: /kaggle/working/OW-DETR/models/deformable_detr_coco_converted.pth


In [4]:
import os
import json

# 6. Thiết lập thư mục dữ liệu VOC2007 và ghi classes.txt (25 lớp)
voc_dir = "data/IP102/VOC2007"
os.makedirs(voc_dir, exist_ok=True)

kaggle_jpeg = "/kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/JPEGImages"
kaggle_xml = "/kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/Annotations"

target_jpeg = os.path.join(voc_dir, "JPEGImages")
target_xml = os.path.join(voc_dir, "Annotations")

if not os.path.exists(target_jpeg):
    os.symlink(kaggle_jpeg, target_jpeg)
    print("Đã symlink thư mục JPEGImages.")

if not os.path.exists(target_xml):
    os.symlink(kaggle_xml, target_xml)
    print("Đã symlink thư mục Annotations.")

# Ghi danh sách 25 lớp vào classes.txt
json_path = "/kaggle/input/datasets/nta212/ip102-for-object-detection/train.json"
with open(json_path, "r", encoding="utf-8") as f:
    coco_data = json.load(f)

categories = sorted(coco_data["categories"], key=lambda x: x["id"])
class_names = [cat["name"] for cat in categories]

classes_file = os.path.join(voc_dir, "classes.txt")
with open(classes_file, "w", encoding="utf-8") as f:
    for name in class_names:
        f.write(f"{name}\n")

print(f"Đã tạo classes.txt với {len(class_names)} lớp sâu bệnh.")


Đã symlink thư mục JPEGImages.
Đã symlink thư mục Annotations.
Đã tạo classes.txt với 25 lớp sâu bệnh.


In [5]:
import os
from pycocotools.coco import COCO

# 7. Chia dữ liệu theo cấu hình mới (Task 1 = 7 lớp: 0..6)
json_path = "/kaggle/input/datasets/nta212/ip102-for-object-detection/train.json"
coco = COCO(json_path)

task_splits = {
    "t1": list(range(0, 7)),
    "t2": list(range(7, 13)),
    "t3": list(range(13, 19)),
    "t4": list(range(19, 25))
}

train_images = {}
for task_name, classes in task_splits.items():
    task_img_ids = set()
    for ann in coco.anns.values():
        if ann["category_id"] in classes:
            task_img_ids.add(ann["image_id"])
    train_images[task_name] = sorted(list(task_img_ids))

all_img_ids = sorted(list(coco.imgs.keys()))
output_dir = "data/IP102/VOC2007/ImageSets/Main"
os.makedirs(output_dir, exist_ok=True)

splits = {
    "t1_train": train_images["t1"],
    "val": all_img_ids,
    "test": all_img_ids,
}

for name, img_ids in splits.items():
    path = os.path.join(output_dir, f"{name}.txt")
    with open(path, "w") as f:
        for img_id in img_ids:
            f.write(f"{img_id:06d}\n" if isinstance(img_id, int) else f"{img_id}\n")
    print(f"Đã tạo file {name}.txt với {len(img_ids)} ảnh.")


loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
Đã tạo file t1_train.txt với 0 ảnh.
Đã tạo file val.txt với 8664 ảnh.
Đã tạo file test.txt với 8664 ảnh.


In [6]:
%cd /kaggle/working/OW-DETR

# 8. Tiến hành huấn luyện Task 1 (7 lớp)
!torchrun --nproc_per_node=2 main_open_world.py \
    --output_dir '/kaggle/working/exps/ip102_t1' \
    --dataset owod \
    --dec_layers 6 \
    --num_queries 100 \
    --batch_size 2 \
    --PREV_INTRODUCED_CLS 0 \
    --CUR_INTRODUCED_CLS 7 \
    --data_root 'data/IP102' \
    --train_set 't1_train' \
    --val_set 'val' \
    --test_set 'test' \
    --num_classes 26 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --pretrain 'models/deformable_detr_coco_converted.pth' \
    --num_workers 4 \
    --eval_every 1 \
    --cache_mode \
    --nc_epoch 0 \
    --epochs 2


/kaggle/working/OW-DETR
W0721 08:45:22.183000 116 torch/distributed/run.py:852] 
W0721 08:45:22.183000 116 torch/distributed/run.py:852] *****************************************
W0721 08:45:22.183000 116 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0721 08:45:22.183000 116 torch/distributed/run.py:852] *****************************************
('aeroplane', 'bicycle', 'bird', 'boat', 'bus', 'car', 'cat', 'cow', 'dog', 'horse', 'motorbike', 'sheep', 'train', 'elephant', 'bear', 'zebra', 'giraffe', 'truck', 'person', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'chair', 'diningtable', 'pottedplant', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'bed', 'toilet', 'sofa', 'frisbee', 'skis', 'snowboard', 'spo

In [7]:
%cd /kaggle/working/OW-DETR

# 9. Đánh giá (Evaluate) mô hình Task 1 sau khi huấn luyện
!torchrun --nproc_per_node=2 main_open_world.py \
    --dataset owod \
    --dec_layers 6 \
    --num_queries 100 \
    --batch_size 10 \
    --PREV_INTRODUCED_CLS 0 \
    --CUR_INTRODUCED_CLS 7 \
    --data_root 'data/IP102' \
    --train_set 't1_train' \
    --val_set 'val' \
    --test_set 'test' \
    --num_classes 26 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --resume '/kaggle/working/exps/ip102_t1/best_checkpoint.pth' \
    --eval


/kaggle/working/OW-DETR
W0721 08:45:36.461000 177 torch/distributed/run.py:852] 
W0721 08:45:36.461000 177 torch/distributed/run.py:852] *****************************************
W0721 08:45:36.461000 177 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0721 08:45:36.461000 177 torch/distributed/run.py:852] *****************************************
('aeroplane', 'bicycle', 'bird', 'boat', 'bus', 'car', 'cat', 'cow', 'dog', 'horse', 'motorbike', 'sheep', 'train', 'elephant', 'bear', 'zebra', 'giraffe', 'truck', 'person', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'chair', 'diningtable', 'pottedplant', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'bed', 'toilet', 'sofa', 'frisbee', 'skis', 'snowboard', 'spo

In [8]:
import shutil
import os

# 10. Nén toàn bộ checkpoint và log của Task 1 thành file zip để tải về
source_dir = "/kaggle/working/exps/ip102_t1"
output_zip = "/kaggle/working/ip102_t1"

if os.path.exists(source_dir):
    print(f"-> Đang nén thư mục checkpoint {source_dir}...")
    shutil.make_archive(output_zip, "zip", source_dir)
    zip_file = output_zip + ".zip"
    size_mb = os.path.getsize(zip_file) / (1024 * 1024)
    print(f"✓ Hoàn thành! File zip được lưu tại: {zip_file} ({size_mb:.2f} MB)")
else:
    print(f"Không tìm thấy thư mục {source_dir}. Vui lòng chạy cell huấn luyện Task 1 trước!")


-> Đang nén thư mục checkpoint /kaggle/working/exps/ip102_t1...
✓ Hoàn thành! File zip được lưu tại: /kaggle/working/ip102_t1.zip (0.00 MB)
